# RAG with LangChain
### Module 5 — LLM Application Frameworks (part 3)

**Goal:** rebuild RAG **with a framework**. In Module 4 you wired retrieve → augment → generate by hand; here LangChain + Chroma do the wiring, closely following Omar Santos's `basic_rag` examples. The knowledge base is **`ai_framework_kb.md`** — cybersecurity AI frameworks (NIST AI RMF, MITRE ATLAS, OWASP Top 10 for LLM Apps, SAIF, CoSAI, the EU AI Act, MAESTRO). This is the foundation for **Mini-Project 1**.

> **Sources:** Omar Santos — repo `part4_rag_examples` (`basic_rag_part1.py`, `basic_rag_part2.py`); book Ch. 2 & 4.

> **Setup:** `pip install langchain langchain-openai langchain-chroma langchain-community unstructured`. Needs `OPENAI_API_KEY` for embeddings + generation. Loading/splitting runs without a key; the embed/store/answer steps are guarded. (Offline alternative noted in the code: swap `OpenAIEmbeddings` for `HuggingFaceEmbeddings`.)

## 🛠️ Setup

In [ ]:
# pip install langchain langchain-openai langchain-chroma langchain-community unstructured python-dotenv
import os
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv())
except Exception:
    pass

HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-4o-mini"
print("OPENAI_API_KEY found:", HAS_KEY)
print("Knowledge base: ai_framework_kb.md")

# 📄 1 — Load & Split the Documents

- Load the knowledge base with a LangChain loader
- Split into chunks with `CharacterTextSplitter`
- `chunk_size` & `chunk_overlap` are the tuning knobs
- Each chunk becomes one entry in the vector store

> ### 🎤 Instructor Script
>
> We start exactly where Omar's basic_rag_part1 starts: load the documents and split them. LangChain's TextLoader reads the file, and CharacterTextSplitter cuts it into chunks of a set size with some overlap — the same idea you coded by hand in Module 4, now one object. This step needs no API key, so it always runs; you'll see how many chunks the framework KB produced. Chunk size and overlap are still the first things to tune if answers come back vague.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import CharacterTextSplitter

loader = TextLoader("ai_framework_kb.md", encoding="utf-8")
documents = loader.load()

splitter = CharacterTextSplitter(chunk_size=900, chunk_overlap=150, separator="\n")
chunks = splitter.split_documents(documents)

print("Loaded", len(documents), "document(s);  split into", len(chunks), "chunks.")
print("\nFirst chunk:\n", chunks[0].page_content[:300], "...")

# 🗄️ 2 — Embed & Store in Chroma

- Embed the chunks with `OpenAIEmbeddings` (text-embedding-3-small)
- Store them in a **Chroma** vector database (persisted to disk)
- `Chroma.from_documents(...)` does embed + store in one call
- Offline option: swap in `HuggingFaceEmbeddings` (no key)

> ### 🎤 Instructor Script
>
> Now we build the vector store, just like Omar does. OpenAIEmbeddings turns each chunk into a vector, and Chroma.from_documents both embeds and stores them, persisting to a folder so you don't rebuild it every run. This is the framework doing in one line what took several cells by hand. If you'd rather stay offline and free, the comment shows how to swap in HuggingFace sentence-transformer embeddings instead — the rest of the code is identical, which is the whole point of LangChain's standard interfaces.

In [ ]:
from langchain_chroma import Chroma

PERSIST_DIR = "db/chroma_ai_frameworks"

if HAS_KEY:
    from langchain_openai import OpenAIEmbeddings
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    # Offline alternative (no key):
    #   from langchain_huggingface import HuggingFaceEmbeddings
    #   embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

    db = Chroma.from_documents(chunks, embeddings, persist_directory=PERSIST_DIR)
    print("Vector store built and persisted to", PERSIST_DIR, "| chunks:", db._collection.count())
else:
    print("[no key] Would embed", len(chunks), "chunks with OpenAIEmbeddings and store them in Chroma.")
    print("Offline option: HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2') — no key needed.")

# 🔍 3 — Make a Retriever

- Turn the vector store into a **retriever** with `.as_retriever(...)`
- Use `similarity_score_threshold` to drop weak matches
- `k` = how many chunks to return
- A retriever is a Runnable — it drops straight into a chain

> ### 🎤 Instructor Script
>
> LangChain wraps search in a retriever object, created with as_retriever. Following Omar's part-2 example, we use a similarity-score-threshold search so only genuinely relevant chunks come back, and k to cap how many. The nice part is that a retriever is itself a Runnable, so it plugs directly into an LCEL chain in the next step — no glue code. Here we run it on a framework question and print the chunks it pulls.

In [ ]:
query = "What are the four core functions of the NIST AI Risk Management Framework?"

if HAS_KEY:
    retriever = db.as_retriever(
        search_type="similarity_score_threshold",
        search_kwargs={"k": 3, "score_threshold": 0.3},
    )
    docs = retriever.invoke(query)
    print("Query:", query, "\n\nRetrieved", len(docs), "chunks:")
    for i, d in enumerate(docs, 1):
        print(f"  [{i}] {d.page_content[:110].strip()}...")
else:
    print("[no key] Would create db.as_retriever(similarity_score_threshold, k=3) and retrieve for:")
    print("   ", query)

# 🔗 4 — Build the RAG Chain (LCEL)

- Combine: retriever → prompt (with context) → model → parser
- `{context}` is filled by the retriever; `{question}` by the user
- One LCEL pipe expresses the whole retrieve-augment-generate flow
- This is Omar's “complete RAG application”

> ### 🎤 Instructor Script
>
> Here's where the framework shines. The entire RAG flow — retrieve, stuff the context into a prompt, call the model, parse the output — becomes one LCEL chain. We use RunnablePassthrough so the user's question flows through untouched while the retriever fills the context slot, then a prompt template, the chat model, and a string parser. This is exactly the pipeline Omar draws in his repo's diagram, written as a single expression. Compare it to the many cells you wrote in Module 4 — same concept, far less wiring.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a cybersecurity assistant. Answer using ONLY the context below. "
               "If the answer isn't in the context, say you don't know.\n\n# Context\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

if HAS_KEY:
    from langchain_openai import ChatOpenAI
    model = ChatOpenAI(model=MODEL, temperature=0)

    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | rag_prompt
        | model
        | StrOutputParser()
    )
    print("RAG chain built:  {context: retriever, question} | prompt | model | parser")
else:
    print("[no key] Built the LCEL RAG chain (retriever -> prompt -> model -> parser).")

# 💬 5 — Ask the Framework Q&A Bot

- Invoke the chain with a plain-English question
- Answers are grounded in the framework documents
- In-domain → grounded answer; off-topic → “I don't know”
- This is your Mini-Project 1 starting point

> ### 🎤 Instructor Script
>
> Finally we use it. Invoke the chain with a question and it retrieves the right framework passages and answers from them. Ask about the NIST functions or the OWASP LLM risks and it answers from the documents; ask something off-topic and, because we told it to, it admits it doesn't know instead of bluffing. That grounded, honest behavior over a real cybersecurity-AI knowledge base is precisely what Mini-Project 1 asks you to build and extend — now you have the framework template to start from.

In [ ]:
questions = [
    "What are the four core functions of the NIST AI Risk Management Framework?",
    "What is MITRE ATLAS used for?",
    "Name three risks from the OWASP Top 10 for LLM Applications.",
]

if HAS_KEY:
    for q in questions:
        print("Q:", q)
        print("A:", rag_chain.invoke(q), "\n")
else:
    print("[no key] Would ask the framework Q&A bot:")
    for q in questions:
        print("  -", q)

# ✅ Summary — RAG, the LangChain Way

- Loader → splitter → embeddings → Chroma → retriever → LCEL chain
- The framework replaces the hand-wiring from Module 4
- Same RAG concepts; far less boilerplate
- Swap embeddings/models with one-line changes
- → Mini-Project 1: extend this over the AI-framework KB

> ### 🎤 Instructor Script
>
> To recap: with LangChain, a full RAG system is a short pipeline — loader, splitter, embeddings, Chroma, retriever, and one LCEL chain — over the cybersecurity-AI-framework knowledge base. It's the same retrieve-augment-generate idea from Module 4, with the framework handling the plumbing and letting you swap embeddings or models in a line. Your first mini-project builds directly on this: extend the knowledge base, tune retrieval, and turn it into a grounded framework assistant with citations. Next we'll look at LangGraph for stateful, multi-step agents.

In [ ]:
print("RAG with LangChain:  load -> split -> embed -> Chroma -> retriever -> (LCEL) prompt|model|parser")
print("Mini-Project 1 starts from this notebook and the ai_framework_kb.md knowledge base.")